In [0]:
from pyspark.sql.functions import current_timestamp, lit, col

In [0]:
def bronze_ingest(table_name):
    (spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("cloudFiles.schemaLocation", f"/Volumes/demo_catalog/bronze/raw_files/_schema/{table_name}")
        .option("header", "true")
        .load(f"/Volumes/demo_catalog/bronze/raw_files/{table_name}/")
        .withColumn("_ingest_timestamp", current_timestamp())
        .withColumn("_source_file", col("_metadata.file_path"))
        .writeStream
        .format("delta")
        .option("checkpointLocation", f"/Volumes/demo_catalog/bronze/raw_files/_checkpoints/{table_name}")
        .trigger(availableNow=True)
        .toTable(f"demo_catalog.bronze.{table_name}_raw")
        .awaitTermination())

In [0]:
for t in ["customers", "products", "orders", "order_items"]:
    bronze_ingest(t)